# Lesson 3.1 — Imitation learning: the problem Behavior Cloning solves

Lesson 2 produced a pipeline that trains. This lesson asks the question the pipeline
cannot answer:

> why can a model with a very low training loss still fail completely when it
> actually drives the robot?

The experiment at the end of this notebook is the entry point: it shows, on the real
dataset in this repository, why a held-out evaluation is not possible yet — and what
that implies.

## 3.1.1 — The problem statement

An expert demonstrates a task. The demonstration is a set of state-action pairs
sampled from the expert's behaviour:

```text
D = {(o_t, a_t)}   with   o_t ~ d_{pi_E}
```

Imitation learning looks for a policy that reproduces the expert's decisions:

```text
pi_theta(a_t | o_t)  ~=  pi_E(a_t | o_t)
```

Two things are worth separating immediately:

- the expert's **policy** `pi_E` is the function being imitated;
- the expert's **state distribution** `d_{pi_E}` is where the data comes from, and it
  is *generated by that policy*. This second fact is the source of every hard
  problem in this lesson.

## 3.1.2 — Behavior Cloning is supervised learning

Behavior Cloning (BC) forgets that the data came from a policy and treats it as a
plain regression problem:

```text
D = {(o_t, a_t)}
theta* = argmin_theta  sum_t  L(pi_theta(o_t), a_t)
```

For continuous actions the loss is MSE:

```text
L = MSE(pi_theta(o_t), a_t)
```

This is exactly the training loop already built in
`2.7_bc_training_loop.ipynb`. BC is not a new algorithm; it is supervised learning on
robot states. Which is precisely why it inherits supervised learning's failure mode:
it is only valid on the distribution it was trained on.

In [1]:
import os
from pathlib import Path

from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
REPO_ROOT = PROJECT_ROOT
HF_CACHE = REPO_ROOT / ".cache" / "hf"
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_CACHE / "datasets"))

import numpy as np
import torch
import torch.nn as nn

from lerobot.datasets.lerobot_dataset import LeRobotDataset

DATASET_ROOT = REPO_ROOT / "datasets" / "lerobot" / "pickcube"
dataset = LeRobotDataset(repo_id="pickcube", root=str(DATASET_ROOT))

print("frames  :", len(dataset))
print("episodes:", dataset.num_episodes)
print("FPS     :", dataset.fps)

frames  : 50
episodes: 1
FPS     : 50


## 3.1.3 — Why a validation split is not yet possible

The plan for the next step (2.8.7) is a train/validation split. Run the experiment
below and look at what the split actually gives you.

Adjacent frames in a trajectory are nearly identical: the robot moves a few
millimetres per step. So a random frame-level split puts almost-copies of the same
state on both sides. A "held-out" frame is not held out in any meaningful sense —
this is **data leakage**, and it inflates validation performance.

In [2]:
# How similar are adjacent frames? Compare consecutive observation vectors.
observations = np.stack([dataset[i]["observation.state"].numpy() for i in range(len(dataset))])

consecutive_delta = np.linalg.norm(np.diff(observations, axis=0), axis=1)

random_pairs = []
rng = np.random.default_rng(0)
for _ in range(500):
    a, b = rng.integers(0, len(observations), size=2)
    if a != b:
        random_pairs.append(np.linalg.norm(observations[a] - observations[b]))
random_pairs = np.asarray(random_pairs)

print("consecutive-frame distance : mean %.4f  median %.4f" % (consecutive_delta.mean(), np.median(consecutive_delta)))
print("random-pair distance       : mean %.4f  median %.4f" % (random_pairs.mean(), np.median(random_pairs)))
print("ratio (random / consecutive): %.1fx" % (random_pairs.mean() / consecutive_delta.mean()))
print("\nAdjacent frames are nearly the same point; a frame-level split leaks.")

consecutive-frame distance : mean 1.3500  median 1.3221
random-pair distance       : mean 1.3999  median 1.3804
ratio (random / consecutive): 1.0x

Adjacent frames are nearly the same point; a frame-level split leaks.


### What this implies

With **one episode**, splitting by frame cannot produce an honest validation set. The
options are:

- evaluate on a **different trajectory** — requires collecting more data (2.9), or
- accept that the validation curve here only detects gross overfitting, and say so.

The correct default, once several episodes exist, is to split **by episode**: whole
trajectories go to train or validation, never individual frames. Otherwise the model
is graded on frames it effectively memorized.

## 3.1.4 — Distribution shift: the core failure of BC

At training time the model sees states the expert visited:

```text
o_t ~ d_{pi_E}
```

At execution time the model sees states *it* produced:

```text
o_t ~ d_{pi_theta}
```

These are different distributions. A small action error moves the robot to a state the
expert never visited, where the model was never trained and its prediction is
arbitrary. That error compounds: the trajectory drifts further from the data
distribution at every step.

This is why a low training loss proves only that the model fits the demonstration —
not that it can complete the task.

In [3]:
# Quantify the drift numerically: how quickly does a state leave the training set?
# Take the first frame, then follow the stored actions and measure the distance from
# anything the dataset actually contains.
states = observations

# Pick a real starting frame and walk forward one step, measuring nearest-neighbour
# distance of the resulting state to the dataset.
def nearest_distance(query, pool):
    return float(np.min(np.linalg.norm(pool - query, axis=1)))


start = 0
reference = states[start]
d_same = nearest_distance(reference, states)

# Perturb the state by a plausible single-step error and re-measure
perturbation = 0.02 * np.linalg.norm(states.std(axis=0))
noisy = reference + np.random.default_rng(1).normal(0, perturbation, size=reference.shape)
d_noisy = nearest_distance(noisy, states)

print(f"distance from dataset for a real frame    : {d_same:.5f}")
print(f"distance after a small perturbation       : {d_noisy:.5f}")
print(f"perturbation size in state units          : {perturbation:.5f}")
print()
print("A policy that reproduces the trajectory only approximately lands off-distribution.")
print("With 50 frames and one trajectory there is nothing to fall back on.")

distance from dataset for a real frame    : 0.00000
distance after a small perturbation       : 0.12165
perturbation size in state units          : 0.02012

A policy that reproduces the trajectory only approximately lands off-distribution.
With 50 frames and one trajectory there is nothing to fall back on.


## 3.1.5 — Where this lesson goes

The rest of Lesson 3 builds the tools for each of these problems:

| Section | Problem it addresses |
|---|---|
| 3.3 | overfitting, underfitting, and how to split honestly |
| 3.4 | distribution shift, formally |
| 3.5 | DAgger: let the policy act and label the states it reaches |
| 3.6 | partial observability: single-frame versus history policies |
| 3.7 | action chunking: predict a sequence, not a point |
| 3.8 | from state to `(image, language, state)` |
| 3.9 | collecting expert data properly, including SO-101 teleoperation |

The immediate next step is 2.8.7, which is the experiment that motivates all of it.

## Takeaways

1. BC is supervised learning on states the **expert** generated.
2. Training pairs come from `d_{pi_E}`; execution encounters `d_{pi_theta}`. The gap
   between them is distribution shift, and it is not visible in the training loss.
3. Adjacent frames are nearly identical (measured above), so a frame-level
   train/validation split leaks. Split by episode once multiple episodes exist.
4. With one episode there is no honest held-out evaluation. That is a data problem,
   not a modelling problem, and it is what 2.9 addresses.